# Data Quality & Anomaly Detector

**The question this tool answers:** *"Did the data feeding my system silently break?"*

Every day, new data flows into systems — customer records, transactions, sensor readings. When something upstream breaks (a column arrives empty, values in the wrong format, numbers suddenly spike), **nothing crashes**. No error appears. The data is just quietly wrong, and everything downstream — dashboards, models, reports — goes wrong with it. Teams often don't notice for days.

This tool is a **smoke detector for data**. It learns what "normal" looks like from a clean reference dataset, then checks each new batch and raises an alarm when something is off.

### The four things it catches
1. **Schema drift** — a column disappeared or an unexpected one appeared.
2. **Null spike** — a column's blank rate jumped far above normal.
3. **Distribution shift** — a numeric column's values moved far from their usual range (e.g. ages suddenly in months instead of years).
4. **Duplicate flood** — far more repeated rows than normal.

> No LLM, no embeddings here. The new idea is **data profiling** — learning a statistical "fingerprint" of clean data, then comparing new data against it. This is core **data engineering**.

Run cells top to bottom with `Shift + Enter`.

## Step 1 — Setup

Just pandas and numpy — standard data tools. Nothing to download, no API key.

In [ ]:
!pip install pandas numpy duckdb -q
import pandas as pd, numpy as np
print("Ready.")

## Step 2 — Create a clean "reference" dataset

In real life this would be a known-good day of data. Here we generate a realistic customer table: `user_id`, `age`, `country`, `email`, and `signup_days_ago`. This clean data is what we'll learn "normal" from.

In [ ]:
np.random.seed(0)

def make_customers(n):
    return pd.DataFrame({
        "user_id": range(n),
        "age": np.random.normal(40, 12, n).clip(18, 90).round(),
        "country": np.random.choice(["US", "UK", "IN", "CA"], n),
        "email": [f"user{i}@example.com" for i in range(n)],
        "signup_days_ago": np.random.randint(0, 730, n),
    })

reference = make_customers(500)
reference.head()

## Step 3 — Learn the "normal" profile

`profile_dataframe` scans the clean data and records its fingerprint: what columns exist, each column's blank rate, the mean/std/min/max of numeric columns, and the normal duplicate rate. This profile is the baseline every future batch is judged against.

In [ ]:
def profile_dataframe(df):
    """Learn what 'normal' looks like: the reference profile of a clean dataset."""
    profile = {
        "columns": list(df.columns),
        "row_count": len(df),
        "null_rates": {c: float(df[c].isna().mean()) for c in df.columns},
        "numeric_stats": {},
        "duplicate_rate": float(df.duplicated().mean()),
    }
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]):
            col = df[c].dropna()
            profile["numeric_stats"][c] = {
                "mean": float(col.mean()) if len(col) else 0.0,
                "std": float(col.std()) if len(col) else 0.0,
                "min": float(col.min()) if len(col) else 0.0,
                "max": float(col.max()) if len(col) else 0.0,
            }
    return profile

profile = profile_dataframe(reference)
print("Normal blank rate for 'age':", round(profile["null_rates"]["age"], 3))
print("Normal mean age:", round(profile["numeric_stats"]["age"]["mean"], 1),
      "+/-", round(profile["numeric_stats"]["age"]["std"], 1))
print("Normal duplicate rate:", round(profile["duplicate_rate"], 3))

## Step 4 — The anomaly checks

`check_batch` compares a new batch against the profile and returns a list of anomalies. Each check has a **threshold** — how far from normal is "too far":
- **null_spike_threshold = 0.20** → flag if blank rate rises more than 20 points.
- **drift_z_threshold = 3.0** → flag if a mean moves more than 3 standard deviations (the classic "3-sigma" rule for outliers).
- **duplicate_threshold = 0.10** → flag if duplicate rate rises more than 10 points.

Thresholds are how you tune sensitivity: lower = pickier (more alerts), higher = more forgiving.

In [ ]:
def check_batch(profile, df, null_spike_threshold=0.20, drift_z_threshold=3.0,
                duplicate_threshold=0.10):
    """Compare a new batch against the reference profile. Returns a list of anomalies."""
    anomalies = []

    # 1) Schema drift — missing or unexpected columns
    expected, actual = set(profile["columns"]), set(df.columns)
    for col in sorted(expected - actual):
        anomalies.append({"type": "Schema drift", "column": col,
            "detail": f"Expected column '{col}' is missing from the new batch.", "severity": "high"})
    for col in sorted(actual - expected):
        anomalies.append({"type": "Schema drift", "column": col,
            "detail": f"Unexpected new column '{col}' appeared.", "severity": "medium"})

    # 2) Null spike — blank rate jumped above normal
    for col in profile["columns"]:
        if col not in df.columns:
            continue
        base = profile["null_rates"].get(col, 0.0)
        now = float(df[col].isna().mean())
        if now - base > null_spike_threshold:
            anomalies.append({"type": "Null spike", "column": col,
                "detail": f"Blank rate in '{col}' rose from {base:.0%} to {now:.0%}.", "severity": "high"})

    # 3) Distribution shift — numeric mean moved many std-devs
    for col, stats in profile["numeric_stats"].items():
        if col not in df.columns or not pd.api.types.is_numeric_dtype(df[col]):
            continue
        new_col = df[col].dropna()
        if len(new_col) == 0:
            continue
        std = stats["std"] if stats["std"] > 0 else 1e-9
        z = abs(float(new_col.mean()) - stats["mean"]) / std
        if z > drift_z_threshold:
            anomalies.append({"type": "Distribution shift", "column": col,
                "detail": f"Average of '{col}' shifted from {stats['mean']:.1f} to {float(new_col.mean()):.1f} ({z:.1f} std-devs).",
                "severity": "high"})

    # 4) Duplicate flood
    dup_rate = float(df.duplicated().mean())
    if dup_rate - profile["duplicate_rate"] > duplicate_threshold:
        anomalies.append({"type": "Duplicate flood", "column": "(whole row)",
            "detail": f"Duplicate-row rate rose to {dup_rate:.0%} (normally {profile['duplicate_rate']:.0%}).",
            "severity": "medium"})

    return anomalies

print("Check function defined.")

## Step 5 — Sanity check: a healthy batch should PASS ✅

First, prove the detector doesn't cry wolf. We feed it a *fresh clean batch* (same generator, different rows). It should find **zero** anomalies.

In [ ]:
healthy_batch = make_customers(400)
healthy_anomalies = check_batch(profile, healthy_batch)
print(f"Healthy batch: {len(healthy_anomalies)} anomalies found.")
print("PASS" if not healthy_anomalies else "unexpected anomalies!")

## Step 6 — Break the data four ways 💥

Now the payoff. We create four broken batches, each simulating a real upstream failure, and confirm the detector catches each one.

In [ ]:
# 1) Schema drift: an upstream change dropped the 'country' column
bad_schema = make_customers(400).drop(columns=["country"])

# 2) Null spike: a bug left 40% of 'age' blank
bad_nulls = make_customers(400)
bad_nulls.loc[bad_nulls.sample(frac=0.4, random_state=1).index, "age"] = np.nan

# 3) Distribution shift: ages recorded in MONTHS instead of years (~x12)
bad_drift = make_customers(400)
bad_drift["age"] = bad_drift["age"] * 12

# 4) Duplicate flood: a re-run appended 200 duplicate rows
bad_dupes = make_customers(400)
bad_dupes = pd.concat([bad_dupes, bad_dupes.head(200)], ignore_index=True)

for name, batch in [("Missing column", bad_schema), ("Null spike", bad_nulls),
                    ("Distribution shift", bad_drift), ("Duplicate flood", bad_dupes)]:
    found = check_batch(profile, batch)
    types = ", ".join(a["type"] for a in found)
    print(f"{name:20} -> caught: {types}")

## Step 7 — A realistic mixed failure + verdict 🚦

Real breakages are often several problems at once. Here a batch has *both* a missing column and a null spike. The tool lists every issue and gives an overall **PASS / FAIL** verdict — the thing a data team would wire to an alert.

In [ ]:
def batch_verdict(anomalies):
    return {"passed": len(anomalies) == 0,
            "count": len(anomalies),
            "high": sum(1 for a in anomalies if a["severity"] == "high")}

mixed_bad = make_customers(400).drop(columns=["email"])
mixed_bad.loc[mixed_bad.sample(frac=0.35, random_state=2).index, "signup_days_ago"] = np.nan

anomalies = check_batch(profile, mixed_bad)
verdict = batch_verdict(anomalies)

print("VERDICT:", "PASS" if verdict["passed"] else "FAIL",
      f"({verdict['count']} anomalies, {verdict['high']} high-severity)\n")
for a in anomalies:
    print(f"  [{a['severity'].upper():6}] {a['type']}: {a['detail']}")

## Step 8 — Save results for the dashboard

We store the reference profile summary and the latest batch's anomalies in a DuckDB file for the Streamlit dashboard.

In [ ]:
import duckdb, json
from datetime import datetime

con = duckdb.connect("dataquality_results.db")
con.execute("CREATE TABLE IF NOT EXISTS anomalies (checked_at TIMESTAMP, type VARCHAR, column_name VARCHAR, detail VARCHAR, severity VARCHAR)")
con.execute("CREATE TABLE IF NOT EXISTS meta (checked_at TIMESTAMP, batch_rows INTEGER, passed BOOLEAN, anomaly_count INTEGER, high_count INTEGER)")
con.execute("DELETE FROM anomalies")
con.execute("DELETE FROM meta")

now = datetime.now()
for a in anomalies:
    con.execute("INSERT INTO anomalies VALUES (?, ?, ?, ?, ?)",
                [now, a["type"], a["column"], a["detail"], a["severity"]])
con.execute("INSERT INTO meta VALUES (?, ?, ?, ?, ?)",
            [now, len(mixed_bad), verdict["passed"], verdict["count"], verdict["high"]])
con.close()
print("Saved to dataquality_results.db")
print("Download it (Files panel) and drop it into the dashboard's data/ folder.")

## ✅ Done — what you built

A data-quality anomaly detector that:
- learns a **statistical profile** of clean data,
- checks each new batch for **schema drift, null spikes, distribution shifts, and duplicate floods**,
- gives a **PASS / FAIL** verdict with per-issue detail,
- and saves results for a dashboard.

**The takeaway to explain in interviews:** bad data breaks pipelines *silently* — no error, just wrong results downstream. This tool is the early-warning system, and it's exactly the kind of reliability infrastructure data teams depend on. It rounds out an "AI/data reliability" portfolio: you monitor the models *and* the data feeding them.

**To make it yours:** swap in a real CSV of your own, tune the thresholds, or add checks (e.g. value-range violations, unexpected categories).

**Next:** download `dataquality_results.db` and open the Streamlit dashboard.